# L12: Explain the Day 2 model globally and locally

**Day 3.** Explanation as an operational requirement: impurity against permutation importance, partial dependence, exact Shapley values by enumeration against a chosen reference, a local surrogate, choosing global or local, and an overfit model that explains itself cleanly and wrongly.

- Reads: checkpoints/L02_clean.csv, checkpoints/L04_split.csv, checkpoints/L06_choice.json
- Writes: work/L12_explanations.json

**Before you start.** Open this notebook from the **Notebooks** folder of the class workspace, next to the **data** and **checkpoints** folders, and run the first two cells.

**How this lab runs.** Work through the cells in order on your own. Each one says what to do with it: **RUN** supplied code, **PREDICT** before running, **DECIDE** by filling a blank, or **WRITE** an answer in words. When most of the room is done, we discuss the answers as a class, then walk through the worked solution together.

### Step 1 · RUN

In [ ]:
# RUN · setup: finds the class files and checks that today's are there. Run it first, every time.
from pathlib import Path
# The class files sit in the folder holding "data" and "checkpoints": this notebook's folder or the one above it.
# (On a local install, a folder named AIDSA in your user folder also works.)
_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents, Path.home() / "AIDSA"] if (p / "data").is_dir() and (p / "checkpoints").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Class files not found: open this notebook from the Notebooks folder of the class workspace.")
DATA, CHECKPOINTS, WORK = ROOT / "data", ROOT / "checkpoints", ROOT / "work"
NEEDED = ["checkpoints/L02_clean.csv",
          "checkpoints/L04_split.csv",
          "checkpoints/L06_choice.json"]
missing = [name for name in NEEDED if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError("Not in the workspace yet: " + ", ".join(missing) + ". Ask your instructor: today's files are not in the workspace yet.")
WORK.mkdir(exist_ok=True)
SOURCE = "checkpoint"  # change to "mine" to use your own saved work where it exists
SEED = 2026
import numpy as np
rng = np.random.default_rng(SEED)

def input_path(name):
    """Where this lab reads an earlier lab's output: your own work folder if you chose "mine" and the file is there,
    otherwise the supplied checkpoint."""
    mine = WORK / name
    return mine if SOURCE == "mine" and mine.exists() else CHECKPOINTS / name

print("Found the class files in:", ROOT, "| start from:", SOURCE)

### Step 2 · RUN

In [ ]:
# RUN · helper functions: the tested course helpers this lab uses. Run it once, after the setup cell.
# You do not need to read them to do the lab; each is explained where the lab uses it.
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import itertools
from math import factorial
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_model(name, preprocessor, seed=2026):
    """A Pipeline of the given preprocessing and one of the course's bounded model families.

    Every family has fixed, small settings so it trains in seconds on a few thousand rows.

    Lab L06.
    """
    models = {
        "logistic regression": LogisticRegression(max_iter=1000),
        "k-nearest neighbours": KNeighborsClassifier(n_neighbors=25),
        "support vector machine": SVC(C=1.0, kernel="rbf", probability=True, random_state=seed),  # probabilities, so every family is read on one scale
        "decision tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=seed),
        "random forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=1, random_state=seed),
        "gradient boosting": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=seed),
    }
    return Pipeline([("prep", preprocessor), ("model", models[name])])


GROUPS = {
    "aircraft": ["aircraft_type", "aircraft_age_years", "total_flight_hours"],
    "maintenance state": ["open_major", "open_minor", "days_since_inspection"],
    "weather": ["temperature_c"],
    "schedule": ["ground_time_hours", "sortie_duration_hours", "mission_type"],
    "load": ["cargo_weight_klb"],
    "base": ["home_base"],
}


def belongs(transformed_name, column):
    """True if a preprocessed feature comes from a raw column: the column itself, one of its one-hot
    categories, or its missing-value indicator.

    Lab L12.
    """
    return (transformed_name == column or transformed_name.startswith(column + "_")
            or transformed_name == f"missingindicator_{column}")


def importance_compare(pipeline, X, y, columns, n_repeats=5, seed=2026):
    """Impurity importance (summed back to raw columns) against permutation importance on held-out rows.

    The pipeline must have steps "prep" (a ColumnTransformer) and "model" (a tree ensemble).

    Lab L12.
    """
    names = pipeline.named_steps["prep"].get_feature_names_out()
    impurity = pd.Series(pipeline.named_steps["model"].feature_importances_, index=names)
    raw = {c: impurity[[n for n in names if belongs(n.split("__", 1)[1], c)]].sum() for c in columns}
    perm = permutation_importance(pipeline, X, y, scoring="roc_auc", n_repeats=n_repeats, random_state=seed, n_jobs=1)
    return pd.DataFrame({"impurity": pd.Series(raw), "permutation": pd.Series(perm.importances_mean, index=X.columns)})


def coalition_value(predict, x_row, background, groups, coalition):
    """Mean prediction when the coalition's groups take x_row's values and the rest come from background rows.

    Lab L12.
    """
    data = background.copy()
    for g in coalition:
        for col in groups[g]:
            data[col] = x_row[col]
    return float(np.mean(predict(data)))


def exact_shapley(predict, x_row, background, groups=GROUPS):
    """Exact Shapley value of every group, by enumerating all coalitions.

    Each group's value is its weighted average marginal contribution v(S with it) - v(S without it).
    The values sum to prediction(x) minus the average prediction on the background (efficiency).

    Lab L12.
    """
    names = list(groups)
    n = len(names)
    v = {frozenset(S): coalition_value(predict, x_row, background, groups, S)
         for size in range(n + 1) for S in itertools.combinations(names, size)}
    phi = {g: 0.0 for g in names}
    for g in names:
        for S in v:
            if g in S:
                continue
            weight = factorial(len(S)) * factorial(n - len(S) - 1) / factorial(n)
            phi[g] += weight * (v[S | {g}] - v[S])
    return phi, v[frozenset()], v[frozenset(names)]


NUMERIC = ["aircraft_age_years", "total_flight_hours", "days_since_inspection", "ground_time_hours",
           "cargo_weight_klb", "temperature_c", "sortie_duration_hours", "open_major", "open_minor"]


CATEGORICAL = ["aircraft_type", "home_base", "mission_type"]


def build_preprocessor(numeric=NUMERIC, categorical=CATEGORICAL):
    """Unfitted preprocessing: numeric median imputation with missing indicators, then scaling;
    categorical one-hot encoding that ignores unseen levels.

    Fit it on training rows only.

    Lab L04.
    """
    numeric_steps = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                              ("scale", StandardScaler())])
    return ColumnTransformer([
        ("num", numeric_steps, numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ])

print("Helper functions ready")

## Part 1: Global explanation

**Specification.** Refit the model chosen on Day 2 and ask what drives it across the whole fleet: impurity importance,
permutation importance, and the partial dependence of risk on temperature.

### Step 3 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
import itertools
import json
from math import factorial

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.inspection import partial_dependence

choice = json.loads(input_path("L06_choice.json").read_text(encoding="utf-8"))
df = pd.read_csv(input_path("L02_clean.csv"), encoding="utf-8")
df = df.merge(pd.read_csv(input_path("L04_split.csv"), encoding="utf-8"), on="mission_id", validate="one_to_one")
cols = NUMERIC + CATEGORICAL
train, test = df[df["split"] == "train"], df[df["split"] == "test"]
model = make_model(choice["model"], build_preprocessor(), seed=SEED).fit(train[cols], train["maint_delay"])
print(f"Day 2 model: {choice['model']}, threshold {choice['threshold']}, test AUC {choice['test_auc']}")

importance = importance_compare(model, test[cols], test["maint_delay"], cols, n_repeats=5, seed=SEED)

### Step 4 · DECIDE

In [ ]:
# DECIDE · DP-L12-1 · which importance measure to rank the features by.
# Replace ____ with your choice. The choices are taught on the slide "Built-in importance, and why it misleads".
# Impurity: the forest's built-in ranking, from training splits. Permutation: the test AUC lost when a column is shuffled.
RANK_BY = ____
print(importance.sort_values(RANK_BY, ascending=False).round(4))

### Step 5 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
pd_result = partial_dependence(model, train[cols], features=["temperature_c"], grid_resolution=30)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(pd_result["values"][0], pd_result["average"][0])
ax.set_xlabel("temperature (C)")
ax.set_ylabel("average predicted risk")
ax.set_title("partial dependence")
plt.show()

### Step 6 · WRITE

**WRITE · your interpretation**. The two importance measures disagree. Which do you trust, and what happened to airframe age and total
flight hours? What shape does the partial dependence on temperature have?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 2: Local explanation

**Specification.** Explain the single test departure the model rates most at risk: an exact Shapley enumeration against a reference
you choose, then a local surrogate. The marginal contribution and the proximity weighting are supplied to read.

### Step 7 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
GROUPS = GROUPS                      # six groups of features, 64 coalitions in all
predict = lambda data: model.predict_proba(data[cols])[:, 1]
scores = predict(test)
row = test.iloc[int(np.argmax(scores))]
x = row[cols]
print(f"{row['mission_id']}: predicted risk {scores.max():.3f}; actually delayed: {bool(row['maint_delay'])}")
print(x.to_string())

### Step 8 · DECIDE

In [ ]:
# DECIDE · DP-L12-2 · the reference the explanation is measured against.
# Replace ____ with your choice. The choices are taught on the slide "The reference is part of the answer".
# "sample": 100 real training departures. "means": one invented departure, every number at its average and every
# category at its most common value.
BACKGROUND = ____
if BACKGROUND == "sample":
    background = train[cols].sample(100, random_state=SEED).reset_index(drop=True)
else:
    background = pd.DataFrame([{c: (train[c].mean() if c in NUMERIC else train[c].mode()[0]) for c in cols}])
print(f"reference: {BACKGROUND}, {len(background)} row(s); average prediction over it {predict(background).mean():.3f}")

### Step 9 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
def shapley(x_row):
    """Exact Shapley value of each group: its weighted average marginal contribution over all coalitions."""
    names = list(GROUPS)
    n = len(names)
    v = {frozenset(S): coalition_value(predict, x_row, background, GROUPS, S)
         for size in range(n + 1) for S in itertools.combinations(names, size)}
    phi = {g: 0.0 for g in names}
    for g in names:
        for S in v:
            if g in S:
                continue
            weight = factorial(len(S)) * factorial(n - len(S) - 1) / factorial(n)
            # The marginal contribution: what adding group g to coalition S changes, weighted by how many
            # orderings put exactly the groups in S before g.
            phi[g] += weight * (v[S | {g}] - v[S])
    return phi, v[frozenset()], v[frozenset(names)]

### Step 10 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
phi, baseline, prediction = shapley(x)
print(pd.Series(phi).sort_values(ascending=False).round(3).to_string())
print(f"baseline {baseline:.3f} + contributions {sum(phi.values()):.3f} = {baseline + sum(phi.values()):.3f} "
      f"(prediction {prediction:.3f})")
reference = exact_shapley(predict, x, background)[0]
print("matches the course helper:", all(abs(phi[g] - reference[g]) < 1e-9 for g in phi))


def surrogate(x_row, n_samples=400, width=0.75):
    """Local surrogate: switch random groups off, weight samples by closeness to x_row, fit a weighted line."""
    from sklearn.linear_model import Ridge
    names = list(GROUPS)
    masks = np.random.default_rng(SEED).integers(0, 2, size=(n_samples, len(names)))
    preds = np.array([coalition_value(predict, x_row, background, GROUPS,
                                                  [g for g, on in zip(names, m) if on]) for m in masks])
    distance = np.sqrt((1 - masks).sum(axis=1) / len(names))   # share of groups switched off
    # The proximity weighting: near samples count most; width sets how local "local" is.
    weights = np.exp(-(distance ** 2) / width ** 2)
    return dict(zip(names, Ridge(alpha=1.0).fit(masks, preds, sample_weight=weights).coef_))

local = surrogate(x)
print(pd.DataFrame({"Shapley": pd.Series(phi), "surrogate": pd.Series(local)}).round(3)
      .sort_values("Shapley", ascending=False).to_string())

### Step 11 · WRITE

**WRITE · your interpretation**. Why was this departure flagged, what did your choice of reference change, what do the Shapley values guarantee
that the surrogate does not, and what should the crew chief be told?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 3: Choosing and doubting

**Specification.** Decide, for three situations, whether a global or a local explanation answers the question. Then choose how deep
a single tree may grow, and watch what its explanation of a wrong prediction looks like.

### Step 12 · WRITE

**WRITE · your interpretation**. For each situation, say global or local, in one sentence each: (1) a crew chief asks why tomorrow's departure on
one tail was flagged; (2) a commander is setting fleet inspection policy; (3) an auditor asks whether home base
influences the model.

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 13 · DECIDE

In [ ]:
# DECIDE · DP-L12-3 · how deep the tree you will explain may grow.
# Replace ____ with your choice. The choices are taught on the slide "When explanations mislead".
# None lets the tree grow until every training departure sits in its own leaf.
MAX_DEPTH = ____
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

deep = Pipeline([("prep", build_preprocessor()),
                 ("model", DecisionTreeClassifier(random_state=SEED, max_depth=MAX_DEPTH))]).fit(train[cols], train["maint_delay"])
p_deep = deep.predict_proba(test[cols])[:, 1]
wrong = np.where((p_deep == 1.0) & (test["maint_delay"].to_numpy() == 0))[0]
print(f"tree, max depth {MAX_DEPTH}: training accuracy {deep.score(train[cols], train['maint_delay']):.3f}, "
      f"test accuracy {deep.score(test[cols], test['maint_delay']):.3f}; "
      f"{len(wrong)} test departures predicted delayed with certainty that left on time")

case = test[cols].iloc[[wrong[0]]]
tree, names = deep.named_steps["model"].tree_, deep.named_steps["prep"].get_feature_names_out()
path = deep.named_steps["model"].decision_path(deep.named_steps["prep"].transform(case)).indices
print(f"why the tree says {test['mission_id'].iloc[wrong[0]]} will certainly be delayed:")
z = deep.named_steps["prep"].transform(case)[0]
for node in path[:-1]:
    f, t = tree.feature[node], tree.threshold[node]
    print(f"  {names[f].split('__', 1)[1]} {'<=' if z[f] <= t else '>'} {t:.2f} (standardised)")
leaf_rows = deep.named_steps["model"].tree_.n_node_samples[path[-1]]
print(f"  -> {len(path) - 1} rules, 100% delay, learned from {leaf_rows} training departure(s)")

### Step 14 · WRITE

**WRITE · your interpretation**. Run the tree with no limit and with a depth of 4. The unlimited tree's explanation is clean, specific and
complete. Why is it worse than no explanation at all, and what should be checked before any model is explained?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 15 · RUN

In [ ]:
# RUN · saves this part's output to work/.
out = {"model": choice["model"], "importance": importance.round(5).to_dict(),
       "local_case": row["mission_id"], "shapley": {k: round(v, 5) for k, v in phi.items()},
       "baseline": round(baseline, 5), "prediction": round(prediction, 5),
       "surrogate": {k: round(float(v), 5) for k, v in local.items()},
       "rank_by": RANK_BY, "background": BACKGROUND, "max_depth": MAX_DEPTH}
(WORK / "L12_explanations.json").write_text(json.dumps(out, indent=2), encoding="utf-8")
print("saved work/L12_explanations.json")